# Titanic Experiment Log Dashboard

This notebook helps you read the project quickly and explore the training log interactively.

It focuses on `outputs/experiment_log.csv`, which is appended every time `main.py` runs. Use the widgets below to filter runs and compare the baseline vs engineered feature scenarios.

In [4]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import Markdown, clear_output, display

LOG_PATH = Path("../outputs/experiment_log.csv")
plt.style.use("seaborn-v0_8-whitegrid")


In [5]:
def load_experiment_log(path=LOG_PATH):
    if not path.exists():
        return pd.DataFrame()

    log_df = pd.read_csv(path)
    if log_df.empty:
        return log_df

    log_df["timestamp_utc"] = pd.to_datetime(log_df["timestamp_utc"], utc=True, errors="coerce")
    log_df = log_df.dropna(subset=["timestamp_utc"]).sort_values("timestamp_utc").reset_index(drop=True)
    log_df["run_number"] = log_df.index + 1
    return log_df


log_df = load_experiment_log()

project_map = pd.DataFrame([
    {"Step": "Load data", "File": "main.py", "What it does": "Reads Titanic CSV and creates train/test splits"},
    {"Step": "Clean data", "File": "src/preprocess.py", "What it does": "Fills missing values and creates engineered features"},
    {"Step": "Train models", "File": "src/train.py", "What it does": "Fits LogisticRegression, RandomForest, GradientBoosting"},
    {"Step": "Evaluate", "File": "src/evaluate.py", "What it does": "Computes accuracy, precision, recall, F1, reports, and plots"},
    {"Step": "Experiment log", "File": "outputs/experiment_log.csv", "What it does": "Tracks best baseline vs engineered results over time"},
])

if log_df.empty:
    display(Markdown("### No experiment log yet\nRun `python main.py` first, then return to this notebook."))
else:
    latest_summary = log_df.iloc[[-1]][[
        "run_number",
        "timestamp_utc",
        "best_baseline_model",
        "best_engineered_model",
        "top_delta_model",
        "top_f1_delta",
        "mean_f1_delta",
    ]].copy()
    latest_summary["timestamp_utc"] = latest_summary["timestamp_utc"].dt.strftime("%Y-%m-%d %H:%M UTC")

    display(Markdown(f"### Loaded {len(log_df)} experiment runs"))
    display(latest_summary)

display(Markdown("### Project map"))
display(project_map)


### Loaded 4 experiment runs

,run_number,timestamp_utc,best_baseline_model,best_engineered_model,top_delta_model,top_f1_delta,mean_f1_delta
3,4,2026-04-18 18:28 UTC,RandomForest,LogisticRegression,LogisticRegression,0.078621,0.038112


### Project map

,Step,File,What it does
0,Load data,main.py,Reads Titanic CSV and creates train/test splits
1,Clean data,src/preprocess.py,Fills missing values and creates engineered fe...
2,Train models,src/train.py,"Fits LogisticRegression, RandomForest, Gradien..."
3,Evaluate,src/evaluate.py,"Computes accuracy, precision, recall, F1, repo..."
4,Experiment log,outputs/experiment_log.csv,Tracks best baseline vs engineered results ove...


In [6]:
if not log_df.empty:
    metric_options = [
        ("Top F1 delta", "top_f1_delta"),
        ("Mean F1 delta", "mean_f1_delta"),
        ("Top accuracy delta", "top_accuracy_delta"),
        ("Mean accuracy delta", "mean_accuracy_delta"),
        ("Best baseline F1", "best_baseline_f1"),
        ("Best engineered F1", "best_engineered_f1"),
    ]

    metric_dropdown = widgets.Dropdown(
        options=metric_options,
        value="top_f1_delta",
        description="Metric:",
        layout=widgets.Layout(width="320px"),
    )

    run_slider = widgets.IntRangeSlider(
        value=(1, int(log_df["run_number"].max())),
        min=1,
        max=int(log_df["run_number"].max()),
        step=1,
        description="Runs:",
        continuous_update=False,
        layout=widgets.Layout(width="520px"),
    )

    output = widgets.Output()

    def render_dashboard(metric, run_window):
        start_run, end_run = run_window
        filtered = log_df[(log_df["run_number"] >= start_run) & (log_df["run_number"] <= end_run)].copy()

        with output:
            clear_output(wait=True)

            if filtered.empty:
                display(Markdown("No experiment rows are available in the selected run range."))
                return

            fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
            fig.suptitle("Titanic Experiment History", fontsize=18, fontweight="bold")

            ax = axes[0]
            ax.plot(filtered["run_number"], filtered["best_baseline_f1"], marker="o", label="Baseline best F1", color="#d95f02")
            ax.plot(filtered["run_number"], filtered["best_engineered_f1"], marker="o", label="Engineered best F1", color="#1b9e77")
            ax.set_title("Best model F1 by run")
            ax.set_ylabel("F1-score")
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)
            ax.legend(loc="lower right")

            ax = axes[1]
            ax.bar(filtered["run_number"], filtered[metric], color="#4c78a8")
            ax.axhline(0, color="black", linewidth=1)
            ax.set_title(f"{metric} across runs")
            ax.set_xlabel("Run number")
            ax.set_ylabel(metric)
            ax.grid(True, axis="y", alpha=0.3)

            plt.show()

            preview_cols = [
                "run_number",
                "timestamp_utc",
                "best_baseline_model",
                "best_engineered_model",
                "top_delta_model",
                metric,
            ]
            preview = filtered[preview_cols].tail(8).copy()
            preview["timestamp_utc"] = preview["timestamp_utc"].dt.strftime("%Y-%m-%d %H:%M UTC")
            display(preview)

    controls = widgets.HBox([metric_dropdown, run_slider])
    display(Markdown("### Interactive experiment dashboard"))
    display(controls, output)

    def _update(change=None):
        render_dashboard(metric_dropdown.value, run_slider.value)

    metric_dropdown.observe(_update, names="value")
    run_slider.observe(_update, names="value")
    _update()
else:
    display(Markdown("### Interactive experiment dashboard\nRun `python main.py` first so this dashboard has a log to display."))


### Interactive experiment dashboard

Output()

## How to read this project

- `main.py` is the main training entrypoint. It compares baseline and engineered features.
- `src/preprocess.py` prepares the Titanic data and adds `FamilySize`, `IsAlone`, and `Title`.
- `src/train.py` trains three models: Logistic Regression, Random Forest, and Gradient Boosting.
- `src/evaluate.py` prints metrics, confusion matrices, feature importance, and subgroup error reports.
- `outputs/experiment_log.csv` keeps a run history. This notebook turns that history into an interactive dashboard.

Tip: use the **Runs** slider to focus on a smaller window and the **Metric** dropdown to change the chart on the lower panel.